# Referenzquelle NIBRS: die gemeinsame Deliktsprache

Diese Quelle liefert keine Zeilen, sondern **Bedeutung**. Chicago und Detroit
zählen dieselben Taten unter verschiedenen Schlüsseln; ohne eine gemeinsame
Klassifikation ist jeder Vergleich eine Behauptung.

**Pipeline-Stufe:** Referenzdokumentation. Es gibt keine Ingestion — die
Zuordnung liegt als zwei gepflegte Tabellen in `config/` und wird in Silver
angewandt.

| | |
| --- | --- |
| Herausgeber | FBI, gemeinsam mit dem Bureau of Justice Statistics |
| Rolle | nationaler Standard der Kriminalitätserfassung der USA seit Januar 2021 |
| Vorgänger | Summary Reporting System (SRS), zum 1.1.2021 eingestellt |
| Im Projekt | Zielsprache für Chicagos IUCR und Detroits MICR |

**Was dieses Notebook zeigt:** dass die Sache eine Falle enthält, die man
übersieht, wenn man nur auf Spaltennamen schaut — und wie die beiden
Mapping-Tabellen des Projekts damit umgehen. Die Quellenverweise stehen am
Ende in Abschnitt 8.

## 1. Was NIBRS ist

**NIBRS** steht für *National Incident-Based Reporting System*. Es ist seit
Januar 2021 der nationale Standard für die Meldung von Kriminalitätsdaten in
den USA und wird von FBI und BJS gemeinsam betrieben.

Der entscheidende Unterschied zum Vorgänger liegt im Namen: **incident-based**.
Das SRS zählte je Vorfall nur die schwerste Straftat — die sogenannte
Hierarchieregel. NIBRS erfasst mehrere Delikte je Vorfall, dazu Angaben zu
Opfern, Tatverdächtigen, Waffen und Sachschaden.

Für dieses Projekt ist das mehr als eine Fussnote: Ein Datensatz mit
Hierarchieregel zählt weniger Delikte als einer ohne, **bei identischer
Kriminalität**. Wer zwei Quellen vergleicht, die diese Regel unterschiedlich
anwenden, vergleicht Erfassungsverfahren und nicht Taten.

Genau deshalb führt das Projekt in Silver `event_key` und `offense_key`
getrennt: Ein Ereignis kann mehrere Delikte tragen.

## 2. Die Falle: Chicagos `fbi_code` ist **nicht** NIBRS

Chicagos Falldaten führen eine Spalte `fbi_code`. Der Name legt nahe, dass dort
der FBI-Standard steht — also NIBRS. Das stimmt nicht: Es ist der **ältere
UCR-/SRS-Schlüssel**.

Die beiden Codesysteme sehen ähnlich aus, sind aber verschiedene Namensräume.
Die nächste Zelle prüft das an den Daten, statt es zu behaupten.

In [ ]:
from pathlib import Path
import csv
import collections

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Projekt-Root mit config/ nicht gefunden')

IUCR_MAPPING = PROJECT_ROOT / 'config' / 'iucr_nibrs_mapping.csv'
KATEGORIE_MAPPING = PROJECT_ROOT / 'config' / 'crime_category_mapping.csv'

iucr_map = list(csv.DictReader(IUCR_MAPPING.open(encoding='utf-8')))
kat_map = list(csv.DictReader(KATEGORIE_MAPPING.open(encoding='utf-8')))

con = duckdb.connect()
CHICAGO = f"read_parquet('{PROJECT_ROOT}/data/bronze/city=chicago/**/*.parquet')"
DETROIT = f"read_json_auto('{PROJECT_ROOT}/data/bronze/detroit/*.ndjson')"

print('IUCR-Mapping   :', len(iucr_map), 'Zeilen')
print('Kategorie-Map  :', len(kat_map), 'Zeilen')
print('Chicago-Bronze :', f"{con.execute(f'SELECT count(*) FROM {CHICAGO}').fetchone()[0]:,}", 'Zeilen')
print('Detroit-Bronze :', f"{con.execute(f'SELECT count(*) FROM {DETROIT}').fetchone()[0]:,}", 'Zeilen')

### 2.1 Zwei Namensräume, keine Überschneidung

In [ ]:
fbi_codes = {z[0] for z in con.execute(
    f'SELECT DISTINCT fbi_code FROM {CHICAGO} WHERE fbi_code IS NOT NULL'
).fetchall()}
ibr_codes = {z['ibr_code'] for z in iucr_map if z['ibr_code'].strip()}

print(f'fbi_code in den Falldaten : {len(fbi_codes):>3} verschiedene')
print('   ', ', '.join(sorted(fbi_codes)))
print()
print(f'ibr_code im Mapping       : {len(ibr_codes):>3} verschiedene')
print('   ', ', '.join(sorted(ibr_codes)))
print()
gemeinsam = fbi_codes & ibr_codes
print('gemeinsame Codes:', sorted(gemeinsam) if gemeinsam else 'KEINE')
print()
print('-> zwei getrennte Codesysteme, nicht zwei Schreibweisen desselben')

### 2.2 Dieselbe Zeile, zwei Schlüssel

Beide Systeme beschreiben oft dasselbe — aber mit anderem Code und anderer
Feinheit. `08B` ist der SRS-Code für einfache Körperverletzung, `13B` der
NIBRS-Code dafür.

In [ ]:
zu_ibr = {z['iucr']: (z['ibr_code'], z['ibr_category']) for z in iucr_map}

print(f"{'IUCR':<6}{'primary_type':<24}{'fbi_code':<10}{'ibr_code':<10}ibr_category")
for iucr, typ, fbi, n in con.execute(f'''
    SELECT iucr, any_value(primary_type), any_value(fbi_code), count(*)
    FROM {CHICAGO} GROUP BY iucr ORDER BY count(*) DESC LIMIT 8
''').fetchall():
    code, kategorie = zu_ibr.get(iucr, ('-', '-'))
    print(f'{iucr:<6}{typ[:22]:<24}{fbi:<10}{code:<10}{kategorie[:30]}')

**Warum das zählt.** Die vergleichbare Kategorie des Projekts wird aus
`fbi_code` gebildet, also aus **SRS**, nicht aus NIBRS — siehe Abschnitt 3.2.
Das NIBRS-Mapping existiert daneben und ist feiner (51 gegen 26 Codes).

Das ist kein Fehler, aber es gehört gewusst: Wer sagt „wir harmonisieren über
NIBRS", beschreibt derzeit die feinere Tabelle, nicht den Weg, den die
Kategorien tatsächlich nehmen.

## 3. Die beiden Mapping-Tabellen des Projekts

Sie tun Verschiedenes und werden leicht verwechselt.

| Datei | Von | Nach | Ebene |
| --- | --- | --- | --- |
| `config/iucr_nibrs_mapping.csv` | Chicagos `iucr` | NIBRS `ibr_code` | fein, 51 Codes |
| `config/crime_category_mapping.csv` | Chicagos `fbi_code`, Detroits `offense_category` | gemeinsame `category` | grob, 28 Kategorien |

### 3.1 `iucr_nibrs_mapping.csv` — Herkunft nachgewiesen

Die Tabelle hat zehn Spalten. **Fünf davon stammen unverändert aus dem
offiziellen Datensatz des Chicago Data Portal**, die anderen fünf sind unsere
Ergänzung. Die nächste Zelle prüft das gegen die Quelle — sie braucht Netz.

In [ ]:
import requests

BASIS_SPALTEN = {'iucr', 'primary_description', 'secondary_description',
                 'index_code', 'active'}
PORTAL = 'https://data.cityofchicago.org/resource/c7ck-438e.json'

try:
    sitzung = requests.Session()
    sitzung.headers['User-Agent'] = 'crime-data-gruppe3/0.1'
    probe = sitzung.get(PORTAL, params={'$limit': 1}, timeout=60)
    probe.raise_for_status()
    anzahl = sitzung.get(PORTAL, params={'$select': 'count(*)'}, timeout=60).json()
    portal_spalten = set(probe.json()[0])
    portal_zeilen = int(anzahl[0]['count'])

    print('Quelle  :', PORTAL)
    print('Zeilen  :', f'im Portal {portal_zeilen}, bei uns {len(iucr_map)}',
          '-> gleich' if portal_zeilen == len(iucr_map) else '-> ABWEICHUNG')
    print('Spalten :', sorted(portal_spalten))
    print('  identisch mit unseren Basisspalten:',
          portal_spalten == BASIS_SPALTEN)
    print()
    print('unsere Ergaenzung:', sorted(set(iucr_map[0]) - portal_spalten))
except requests.RequestException as fehler:
    print('Portal nicht erreichbar:', fehler)
    print('unsere Spalten:', sorted(iucr_map[0]))

### 3.2 `crime_category_mapping.csv` — die vergleichbare Ebene

Hier laufen beide Städte zusammen. Chicago tritt über `fbi_code` ein, Detroit
über `offense_category`.

In [ ]:
je_stadt = collections.defaultdict(list)
for z in kat_map:
    je_stadt[z['city']].append(z)

for stadt, zeilen in sorted(je_stadt.items()):
    felder = {z['source_field'] for z in zeilen}
    print(f'{stadt:<9}{len(zeilen):>3} Zeilen   Eintrittsfeld: {felder}')

kategorien = {z['category'] for z in kat_map}
vergleichbar = {z['category'] for z in kat_map
                if z['comparable'].strip().lower() == 'true'}
in_beiden = {
    k for k in vergleichbar
    if {z['city'] for z in kat_map
        if z['category'] == k and z['comparable'].strip().lower() == 'true'}
    == {'chicago', 'detroit'}
}
print()
print(f'Kategorien insgesamt      : {len(kategorien)}')
print(f'als vergleichbar markiert : {len(vergleichbar)}')
print(f'in beiden Staedten belegt : {len(in_beiden)}')
print()
print('  ', ', '.join(sorted(vergleichbar)))

**Nur eine Kategorie, die in beiden Städten belegt ist, trägt einen Vergleich.**
Deshalb ist diese Zahl — und nicht die Zahl aller Kategorien — die Grundlage der
Gold-Schicht.

## 4. Woher der Crosswalk kommt, und was daran Handarbeit ist

Die Spalte `match` hält fest, wie jede Zeile zu ihrem NIBRS-Code kam. Das ist
die ehrlichste Angabe der Tabelle, denn sie zeigt, wie viel **nicht** aus einer
amtlichen Quelle stammt.

In [ ]:
herkunft = collections.Counter(z['match'] or '(leer)' for z in iucr_map)
gesamt = len(iucr_map)

erklaerung = {
    'isp_direct': 'direkt aus der Codetabelle der Illinois State Police',
    'isp_base_code': 'ueber den Basiscode der ISP-Tabelle abgeleitet',
    'manual': 'von Hand zugeordnet',
    'manual_override': 'von Hand korrigiert, gegen die abgeleitete Zuordnung',
    '(leer)': 'keine Zuordnung',
}
print(f"{'Herkunft':<18}{'Zeilen':>8}{'Anteil':>9}   Bedeutung")
for schluessel, n in herkunft.most_common():
    print(f'{schluessel:<18}{n:>8}{n / gesamt:>9.1%}   {erklaerung.get(schluessel, "")}')

handarbeit = herkunft['manual'] + herkunft['manual_override']
print()
print(f'Handarbeit insgesamt: {handarbeit} von {gesamt} = {handarbeit / gesamt:.0%}')

**Zwei Fünftel der Tabelle sind Handarbeit.** Das ist keine Schwäche, solange es
dabeisteht: Für viele IUCR-Codes existiert keine amtliche Eins-zu-eins-Zuordnung,
weil Illinois eigene Straftatbestände führt. Aber es heisst auch, dass die
Aussage „vollständig auf NIBRS abgebildet" eine **gepflegte Behauptung** ist und
kein amtlicher Nachweis.

Was im Repository fehlt: ein Verweis darauf, **welche** Fassung der
ISP-Codetabelle verwendet wurde. Die Tabelle wird jährlich neu veröffentlicht
(Abschnitt 8), und ohne Jahresangabe ist die Herkunft nicht nachvollziehbar.

## 5. Abdeckung: was zugeordnet ist und was nicht

In [ ]:
ohne_zuordnung = [z for z in iucr_map if not z['ibr_code'].strip()]
print(f'ohne NIBRS-Zuordnung: {len(ohne_zuordnung)} von {len(iucr_map)}')
for z in ohne_zuordnung:
    print(f"  {z['iucr']}  {z['primary_description']} / {z['secondary_description']}"
          f"   (aktiv={z['active']})")

if ohne_zuordnung:
    codes = tuple(z['iucr'] for z in ohne_zuordnung)
    treffer = con.execute(
        f'SELECT count(*) FROM {CHICAGO} WHERE iucr IN {codes}'
    ).fetchone()[0]
    print(f'\n  davon in den Falldaten: {treffer}')

in_daten = {z[0] for z in con.execute(
    f'SELECT DISTINCT iucr FROM {CHICAGO}').fetchall()}
bekannt = {z['iucr'] for z in iucr_map}
print(f'\nverschiedene IUCR-Codes in den Falldaten: {len(in_daten)}')
print('nicht in der Mapping-Tabelle:', sorted(in_daten - bekannt) or 'keiner')

**Der Unterschied zwischen zwei Aussagen** ist hier wichtig:

- „Die Mapping-Tabelle ist vollständig" — **falsch**, zwei Codes sind offen.
- „Jeder Code, der in den Falldaten vorkommt, ist zugeordnet" — **richtig**.

Für H3 zählt die zweite. Für die Aussage über die Tabelle die erste. Wer beides
vermischt, behauptet mehr als er gemessen hat.

## 6. Detroits Seite: MICR

Michigan meldet über **MICR**, den *Michigan Incident Crime Reporting*-Standard.
Detroits Falldaten führen ihn als `state_offense_code`, daneben eine gröbere
`offense_category`.

Das Projekt bildet auf der **gröberen** Ebene ab. Die nächste Zelle zeigt, was
dabei ungenutzt bleibt.

In [ ]:
zahlen = con.execute(f'''
    SELECT count(DISTINCT attributes.offense_category),
           count(DISTINCT attributes.state_offense_code),
           count(DISTINCT trim(attributes.offense_description))
    FROM {DETROIT}
''').fetchone()
print(f'offense_category    {zahlen[0]:>4}   <- hierauf bildet das Mapping ab')
print(f'state_offense_code  {zahlen[1]:>4}   <- MICR, feiner, derzeit ungenutzt')
print(f'offense_description {zahlen[2]:>4}')

print('\nMICR-Codes je Kategorie, die groessten:')
for kat, codes, n in con.execute(f'''
    SELECT attributes.offense_category, count(DISTINCT attributes.state_offense_code),
           count(*)
    FROM {DETROIT} GROUP BY 1 ORDER BY 2 DESC LIMIT 5
''').fetchall():
    print(f'  {kat[:30]:<30}{codes:>3} Codes  {n:>7,} Faelle')

### 6.1 Ist die feinere Ebene brauchbar?

Zwei verschiedene Fragen, die man leicht vermischt:

1. **Trägt ein MICR-Code immer dieselbe Kategorie?** Wenn nicht, ist die
   Zuordnung Code → Kategorie mehrdeutig.
2. **Trägt ein MICR-Code immer dieselbe Beschreibung?** Wenn nicht, ist die
   Erfassung innerhalb einer Kategorie uneinheitlich.

Die Zelle prüft beides getrennt.

In [ ]:
mehrdeutig = con.execute(f'''
    SELECT attributes.state_offense_code AS code,
           count(DISTINCT attributes.offense_category) AS kategorien,
           count(*) AS zeilen
    FROM {DETROIT} WHERE attributes.state_offense_code IS NOT NULL
    GROUP BY 1 HAVING count(DISTINCT attributes.offense_category) > 1
    ORDER BY zeilen DESC
''').fetchall()
print(f'MICR-Codes mit mehr als einer Kategorie: {len(mehrdeutig)} von {zahlen[1]}')
for code, kategorien, zeilen in mehrdeutig:
    print(f'  {code}: {kategorien} Kategorien, {zeilen:,} Zeilen')

print('\nZweite Frage - derselbe Code, verschiedene Beschreibungen (MICR 2301):')
for kat, beschr, n in con.execute(f'''
    SELECT attributes.offense_category, trim(attributes.offense_description), count(*)
    FROM {DETROIT} WHERE attributes.state_offense_code = '2301'
    GROUP BY 1,2 ORDER BY 3 DESC
''').fetchall():
    print(f'  {kat:<10}{beschr[:44]:<46}{n:>6,}')

**Die feinere Ebene ist fast sauber.** Nur zwei von 97 Codes überspannen
Kategorien, und bei den Beschreibungen sind die Abweichungen Einzelfälle. MICR wäre also nutzbar — die
Entscheidung für die gröbere Ebene ist eine Abwägung und keine Notwendigkeit.

Was dagegen spricht, MICR jetzt zu nutzen: Für Chicago müsste dann ebenfalls die
feinere Ebene gelten, also IUCR statt `fbi_code`, und damit stünde die
NIBRS-Tabelle mit ihren zwei Fünfteln Handarbeit im Weg zur Vergleichszahl statt
daneben.

### 6.2 Eine Bereinigungsaufgabe, die hier sichtbar wird

In [ ]:
roh, getrimmt, gepolstert, detroit_zeilen = con.execute(f'''
    SELECT count(DISTINCT attributes.offense_description),
           count(DISTINCT trim(attributes.offense_description)),
           count(*) FILTER (attributes.offense_description
                            <> trim(attributes.offense_description)),
           count(*)
    FROM {DETROIT}
''').fetchone()

print(f'Zeilen mit Randleerzeichen: {gepolstert:,} von {detroit_zeilen:,} '
      f'= {gepolstert / detroit_zeilen:.1%}')
print()
print(f'verschiedene Beschreibungen, ungetrimmt: {roh}')
print(f'verschiedene Beschreibungen, getrimmt  : {getrimmt}')
print(f'-> {roh - getrimmt} Scheinwerte allein durch Leerzeichen')
print()
print('Bronze laesst den Text wie geliefert; das Trimmen gehoert nach Silver.')

**Fast die Hälfte der Beschreibungen ist ein Artefakt.** Ungetrimmt zählt die
Quelle 151 verschiedene Deliktbeschreibungen, getrimmt nur 81 — **70 Werte
entstehen allein durch Leerzeichen**.

Wer ohne `trim()` gruppiert, bekommt dieselbe Beschreibung zweimal und hält den
Unterschied womöglich für eine Erfassungsvariante. Das ist genau die Sorte
Fehler, die in einer Auswertung nicht auffällt, weil das Ergebnis plausibel
aussieht.

Bronze lässt den Text wie geliefert — das ist richtig, denn die Auffüllung
gehört zur Quelle. Das Trimmen gehört nach Silver, zusammen mit dem Parsen der
Zeitstempel und dem Auflösen der Fehlwerte.

## 7. Was das für H3 bedeutet

**H3 lautet:** Die Deliktschlüssel beider Städte lassen sich vollständig auf
NIBRS abbilden.

Nach diesem Notebook ist die Hypothese zu schärfen, bevor sie geprüft wird:

| Frage | Antwort |
| --- | --- |
| Ist Chicagos IUCR vollständig auf NIBRS abgebildet? | in der Tabelle nein (2 offen), in den Falldaten ja |
| Ist Detroits MICR auf NIBRS abgebildet? | **nein** — es gibt keine MICR-NIBRS-Tabelle im Repository |
| Laufen beide Städte über NIBRS zusammen? | **nein** — die vergleichbare Kategorie kommt aus `fbi_code` (SRS) und `offense_category` |

Der dritte Punkt ist der wesentliche. Das Projekt harmonisiert heute über eine
**eigene Kategorienliste**, nicht über NIBRS. NIBRS ist als feinere Zuordnung
für Chicago vorhanden, trägt aber die Vergleichszahl nicht.

Das ist ein vertretbarer Entwurf — er braucht nur die richtige Bezeichnung. Wer
im Kolloquium sagt „wir harmonisieren über NIBRS", bekommt die Rückfrage, wo
Detroits NIBRS-Codes herkommen. Die Antwort wäre heute: nirgendwoher.

## 8. Quellenverweise

**Standard und Herausgeber**

| Was | Adresse |
| --- | --- |
| NIBRS beim Bureau of Justice Statistics | <https://bjs.ojp.gov/national-incident-based-reporting-system-nibrs> |
| NIBRS beim FBI | <https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr/nibrs> |
| *A Guide to Understanding NIBRS* (FBI, PDF) | <https://www.fbi.gov/file-repository/ucr/a-guide-to-understanding-nibrs.pdf> |
| CRS-Bericht R46668, mit Gegenüberstellung der SRS- und NIBRS-Deliktlisten | <https://www.congress.gov/crs-product/R46668> |

**Illinois und Chicago**

| Was | Adresse |
| --- | --- |
| IUCR-Codes, Datensatz des Chicago Data Portal (`c7ck-438e`) | <https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e> |
| Crime Reporting der Illinois State Police | <https://isp.illinois.gov/CrimeReporting> |
| NIBRS-Codetabellen der ISP, jährliche Fassung | <https://isp.illinois.gov/StaticFiles/docs/CrimeReporting/nibrs/2025%20nibrs_CodesTables.xlsx> |
| Illinois-UCR-Portal | <https://ilucr.nibrs.com/> |

**Im Repository**

| Was | Wo |
| --- | --- |
| IUCR nach NIBRS | `config/iucr_nibrs_mapping.csv` |
| Beide Städte nach gemeinsamer Kategorie | `config/crime_category_mapping.csv` |
| Konzeptioneller Vorlauf zu dieser Frage | [`docs/nibrs_chicago_detroit_abgleichen.md`](../docs/nibrs_chicago_detroit_abgleichen.md) |
| Abgabetext, Kapitel 2 und 3.3 | [`docs/ln1_konzept.md`](../docs/ln1_konzept.md) |

**Stand der Abrufe:** Die Angaben zu NIBRS und zum Chicago Data Portal wurden am
10.9.2026 geprüft. Die Seite des FBI antwortete auf automatisierte Abrufe mit
HTTP 403; sie ist im Browser erreichbar und hier als Verweis aufgeführt, nicht
als geprüfte Quelle.

## 9. Offen

- **Für Detroit fehlt eine MICR-NIBRS-Tabelle.** Ohne sie ist H3 für Detroit
  nicht prüfbar.
- **Die Fassung der ISP-Codetabelle ist nicht festgehalten**, obwohl sie
  jährlich wechselt.
- **Zwei IUCR-Codes sind ohne Zuordnung.** Beide kommen 2024 nicht vor; einer
  ist ohnehin inaktiv.
- **Die Benennung der Harmonisierung** sollte dem folgen, was tatsächlich
  geschieht — heute eine eigene Kategorienliste über SRS und `offense_category`,
  nicht NIBRS.

In [ ]:
con.close()
print('Verbindung geschlossen.')